# ICU glycemic harm in the next 24 hours: the tutorial, start to finish

One notebook, four stages, run top to bottom in Google Colab on a CPU runtime. Everything a peer needs to re-run it is in `docs/runbook.md`; the short version:

- **Stage 0, setup.** Mounts your Drive (one Google account for everything: the one linked on PhysioNet), clones the code from the GitHub repository named in `REPO` (or, with `REPO` empty, takes `hypo-harm-tutorial.zip` from `MyDrive/hypo_tutorial/` or asks for it), refuses code from another release, and decides whether the data stage has to run.
- **Stage 1, data.** The SQL on BigQuery in your own project: the extract once (29.1 GiB of query bytes, 2.47 GiB of storage, measured), the pipeline (`01_cohort_labels.sql`, `02_features.sql`, about 11 GiB), the pull of three parquet files to your Drive. Skipped automatically when `features.parquet` is already there; `RUN_DATA_STAGE` in the setup cell forces or forbids it.
- **Stage 2, cohort and labels.** The pulled rows checked the way the SQL's own tables check them: sizes and rates per split, era and ICU day, censoring, the leakage assertion on every row, missingness and monitoring density.
- **Stage 3, the models.** For each outcome (glucose < 54 and > 300 mg/dL in the next 24 h of ICU time) a logistic regression (arm A) and a LightGBM model (arm B), trained on the patients anchored in 2008 to 2016, scored on those anchored in 2017 to 2019 (the primary test) and in 2020 to 2022 (the documentation-shift test); the split is patient-disjoint and approximately temporal (a patient's later admissions keep the group), with subject-grouped cross-validation, Platt recalibration, subject-level bootstrap intervals, calibration, decision curves, operating points, subgroups, SHAP and three sensitivities.
- **Stage 4, pack.** One zip of `results/` (aggregates only) to your computer. `parquet/` and `predictions/` hold MIMIC-IV identifiers and stay on your Drive.

Every number a reader could question has a procedure behind it: preprocessing statistics from the training rows only; cross-validation grouped by subject and stratified by outcome; log loss as the tuning criterion because the outputs are read as risks; intervals that resample subjects, not rows; the three families TRIPOD+AI asks for (discrimination, calibration, clinical utility); subgroup tables written to be shared (levels under 100 rows pooled, cells under 10 events without a count).

References read for the design: Collins GS et al., TRIPOD+AI statement, BMJ 2024;385:e078378. Van Calster B et al., Calibration: the Achilles heel of predictive analytics, BMC Medicine 2019;17:230. Vickers AJ, Elkin EB, Decision curve analysis, Medical Decision Making 2006;26:565. Riley RD et al., Minimum sample size for developing a multivariable prediction model, Statistics in Medicine 2019;38:1276. Ke G et al., LightGBM, NeurIPS 2017. Lundberg SM et al., consistent individualized feature attribution for tree ensembles (TreeSHAP), 2018.

In [ ]:
# 0. Setup: Drive, the code, where things are, and the three switches
import os, sys, json, glob, pathlib, shutil, zipfile, subprocess, warnings, importlib.metadata
warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT = ''              #@param {type:"string"}
REPO = ''                 #@param {type:"string"}
RUN_DATA_STAGE = 'auto'   #@param ["auto", "always", "never"]
EXTRACT_FORCE = False     #@param {type:"boolean"}
# PROJECT: your Google Cloud project id (console.cloud.google.com, top bar; the id, not the display name), used by the
#          data stage only.
# REPO:    the GitHub repository this notebook came from, https://github.com/<account>/<repository>; the code (SQL, the
#          runner, the Python package) is cloned from it. Leave it empty to use hypo-harm-tutorial.zip from your Drive.
# RUN_DATA_STAGE: auto runs BigQuery only when features.parquet is not on your Drive yet.
# EXTRACT_FORCE: rewrites the src_* extract when the project already holds one. Needed once on a project whose extract
#          predates release .20 (the runner says so); it spends about 2.8 GiB of a sandbox's lifetime storage, nothing
#          on a project with billing. Leave it off otherwise.

IN_COLAB = os.path.exists('/content')
RELEASE = '2026-10-12.22'                   # this notebook belongs to this release of the repository (its RELEASE file, first line)
if IN_COLAB:
    # Pick ONE Google account in the Drive dialog, the one linked on PhysioNet, and use it every time: the parquet and
    # the results live in that account's Drive, and another account mounts an empty folder (the check below catches it)
    from google.colab import drive
    drive.mount('/content/drive')
    DEST = pathlib.Path('/content/drive/MyDrive/hypo_tutorial')
    DEST.mkdir(parents=True, exist_ok=True)
    print('MyDrive/hypo_tutorial holds:', sorted(p.name for p in DEST.iterdir()) or 'nothing yet (first run on this account)')
    ROOT = pathlib.Path('/content/hypo-harm-tutorial')
    if ROOT.exists():
        shutil.rmtree(ROOT)
    if REPO:
        # the code comes from the repository the notebook was opened from; the notebook and the repository must be the
        # same release (the RELEASE file's first line), or stage 3 could import code the notebook was not written for
        r = subprocess.run(['git', 'clone', '--quiet', '--depth', '1', REPO, str(ROOT)], capture_output=True, text=True)
        if r.returncode != 0 or not (ROOT / 'RELEASE').exists():
            raise SystemExit(f'could not clone {REPO}: {r.stderr.strip()[:300]}\n'
                             'Check the address and that the repository is public; or leave REPO empty and use the zip from your Drive.')
        got = (ROOT / 'RELEASE').read_text().split()[0]
        if got != RELEASE:
            raise SystemExit(f'this notebook is release {RELEASE} but the repository is at release {got}: open the notebook '
                             'from the repository again (File, Open notebook, GitHub tab) so the two match')
        print(f'code cloned from {REPO}, release {got}')
    else:
        ZIP = DEST / 'hypo-harm-tutorial.zip'
        def is_current(path):
            with zipfile.ZipFile(path) as z:
                names = z.namelist()
                return 'hypo-harm-tutorial/RELEASE' in names and z.read('hypo-harm-tutorial/RELEASE').decode().split()[0] == RELEASE
        if not ZIP.exists() or not is_current(ZIP):
            from google.colab import files
            print(f'hypo-harm-tutorial.zip is not on Drive yet, or is an older release than {RELEASE}; pick the new one from your computer:')
            picked = next(iter(files.upload()))
            shutil.copy(picked, ZIP)
        with zipfile.ZipFile(ZIP) as z:
            z.extractall('/content')
        print(f'code unpacked from {ZIP}, release {RELEASE}')
    PARQUET, RESULTS, PREDICTIONS = DEST / 'parquet', DEST / 'results', DEST / 'predictions'
else:
    # outside Colab (a laptop, after pipeline/run_bigquery.py --step pull): the repository is the parent of
    # notebooks/, HYPO_PARQUET names the folder with the three parquet files, and the data stage never runs here
    ROOT = pathlib.Path('..').resolve()
    if 'HYPO_PARQUET' not in os.environ:
        raise SystemExit('outside Colab, set HYPO_PARQUET to the folder holding params, cohort_landmarks and features.parquet')
    PARQUET = pathlib.Path(os.environ['HYPO_PARQUET'])
    RESULTS = pathlib.Path(os.environ.get('HYPO_RESULTS', str(PARQUET.parent / 'results')))
    PREDICTIONS = PARQUET.parent / 'predictions'
for d in (PARQUET, RESULTS, RESULTS / 'figures', PREDICTIONS):
    d.mkdir(parents=True, exist_ok=True)
NEED_DATA = RUN_DATA_STAGE == 'always' or (RUN_DATA_STAGE == 'auto' and not (PARQUET / 'features.parquet').exists())

# the two modeling libraries, pinned to the versions the tutorial was run with: Colab ships its own lightgbm, and a
# different version gives slightly different trees, so the pinned one is installed whenever the installed one differs
for pkg, mod in [('lightgbm==4.7.0', 'lightgbm'), ('shap==0.53.0', 'shap')]:
    want = pkg.split('==')[1]
    try:
        have = importlib.metadata.version(mod)
    except importlib.metadata.PackageNotFoundError:
        have = None
    if have != want:
        print(f'installing {pkg} (found {have})')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from hypo_tutorial import columns as C, features as F, models as M, evaluate as E
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
print('code:', ROOT, '| parquet:', PARQUET, '| results:', RESULTS, '| predictions:', PREDICTIONS)
present = sorted(PARQUET.glob('*.parquet'))
print('parquet files:', ', '.join(f'{f.name} ({f.stat().st_size / 1e6:.1f} MB)' for f in present) or 'NONE')
print('data stage:', 'will run on BigQuery' if NEED_DATA else 'skipped, features.parquet is present (RUN_DATA_STAGE = always forces it)')

## 1. Data stage: the SQL on BigQuery, once

MIMIC-IV v3.1 already sits on BigQuery for every credentialed PhysioNet user, so nothing is downloaded and no file is edited: the SQL files run as they are in your own Google Cloud project, and the runner (`pipeline/run_bigquery.py`) fills in the project id. Three steps, each one call to the runner:

1. **extract**: `sql/00b_extract.sql` runs once as a script job and writes the `src_*` tables (measured on the first real run: 29.1 GiB of the free 1 TiB of query bytes a month, 2.47 GiB of the sandbox's lifetime 10 GiB of storage). The runner refuses to run it a second time on a project that already holds a complete extract, and tells you to add `--force` after one that stopped partway.
2. **pipeline**: `01_cohort_labels.sql` (twenty-one check tables, saved as `01_cohort_labels_table01.csv` to `table21.csv`) and `02_features.sql` (the leakage assertion, which must say 0, Table 1b and the profile, `02_features_check01.csv` to `check03.csv`), about 6 GiB each; repeatable, no storage beyond the two landmark tables (tens of MB).
3. **pull**: `params`, `cohort_landmarks` and `features` to parquet (about 3 GiB, 0 from the result cache), copied to `MyDrive/hypo_tutorial/parquet/` with the check tables to `results/`.

These cells run only when the setup cell decided the data stage is needed; otherwise each prints one line and the notebook goes on to stage 2 with the parquet already on Drive.

In [ ]:
# 1.1 Sign in to BigQuery with the Google account linked on PhysioNet, and the month's counters
def run_step(step, *extra):
    cmd = [sys.executable, str(ROOT / 'pipeline' / 'run_bigquery.py'), '--project', PROJECT, '--step', step, *extra]
    lines = []
    with subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end=''); lines.append(line)
    return p.returncode, ''.join(lines)

if NEED_DATA:
    assert IN_COLAB, 'the data stage signs in to BigQuery from Colab; outside Colab run pipeline/run_bigquery.py directly'
    assert PROJECT, 'paste your project id into PROJECT in the setup cell and run it again'
    from google.colab import auth
    auth.authenticate_user()
    code_, _ = run_step('cost')              # the month's query bytes against the free 1 TiB, the dataset's GiB against the lifetime 10 GiB
    if code_:
        raise RuntimeError('the cost check failed; the output above says why (an unknown project id, or an account without BigQuery access)')
else:
    print('data stage skipped')

In [ ]:
# 1.2 Extract, once per project (several minutes)
if NEED_DATA:
    code_, text = run_step('extract', *(['--force'] if EXTRACT_FORCE else []))
    if code_ and 'already exist' in text:
        print('\nthe extract is already in this project; going on')
    elif code_:
        raise RuntimeError('the extract failed; the output above says why. An extract from before release .20, or one that stopped partway, is rewritten by setting EXTRACT_FORCE to True in the setup cell for one run (docs/runbook.md, section 6)')
else:
    print('data stage skipped')

In [ ]:
# 1.3 Pipeline: views, parameters, landmarks and the check tables (about two and a half minutes)
if NEED_DATA:
    code_, _ = run_step('pipeline')
    if code_:
        raise RuntimeError('the pipeline failed; the output above says why')
else:
    print('data stage skipped')

In [ ]:
# 1.4 Pull the final tables to parquet and keep everything on Drive
if NEED_DATA:
    code_, _ = run_step('pull', '--parquet-dir', '/content/parquet')
    if code_:
        raise RuntimeError('the pull failed; the output above says why')
    for f in glob.glob('/content/parquet/*.parquet'):
        shutil.copy(f, PARQUET / os.path.basename(f))
    for f in glob.glob(str(ROOT / 'results' / '**' / '*.csv'), recursive=True):     # the check tables of 01 and 02 (and the extract's size table)
        shutil.copy(f, RESULTS / os.path.basename(f))
    print(f"{sum(f.stat().st_size for f in PARQUET.glob('*.parquet')) / 2**20:,.0f} MB of parquet and "
          f"{len(list(RESULTS.glob('0*_*.csv')))} check tables under MyDrive/hypo_tutorial/")
    for f in sorted(PARQUET.glob('*.parquet')):
        print(f'  {f.name}: {f.stat().st_size / 1e6:.1f} MB')
else:
    print('data stage skipped')

## 2. Cohort and labels: what the data stage produced

Reads the parquet files and checks them the way the SQL's own check tables do, then adds what SQL does not show well: the censoring profile, the leakage assertion on every row, feature missingness and monitoring density. Every number here is an aggregate; no row of the data is printed.

In [ ]:
# 2.0 Load the pulled rows
assert (PARQUET / 'features.parquet').exists(), f'features.parquet is not in {PARQUET}: run the data stage on this Drive first (RUN_DATA_STAGE = always)'
land = pd.read_parquet(PARQUET / 'cohort_landmarks.parquet')
feat = pd.read_parquet(PARQUET / 'features.parquet')
params = pd.read_parquet(PARQUET / 'params.parquet')
LABELS = C.LABELS                 # y_lt54_24h (primary: any documented < 54), y_lt54_excl_24h (the repeat-excluded sensitivity endpoint), y_lt40_24h, y_cms816_24h, y_gt300_24h
print(f'landmark rows {len(land):,}   feature rows {len(feat):,} x {feat.shape[1]} columns')
print('parameters the SQL ran with:'); print(params.T.rename(columns={0: 'value'}))

## 2.1 Cohort size and event rates

The same numbers as table 1 of `01_cohort_labels.sql`, recomputed from the pulled rows. If they differ from `results/01_cohort_labels_table01.csv`, the pull is incomplete. `pos_lt54` counts every documented result below 54 in the window; `pos_lt54_excl` the same after the CMS816 repeat-result exclusion (a sensitivity endpoint, not the primary one: a contradicted low is excluded by the rule, not shown to be a measurement error).

In [ ]:
# 2.1 Rows, stays, admissions, subjects and events per split
def rates(df, by):
    g = df.groupby(by, observed=True)
    out = pd.DataFrame({'rows': g.size(), 'stays': g['stay_id'].nunique(), 'admissions': g['hadm_id'].nunique(),
                        'subjects': g['subject_id'].nunique()})
    for y in LABELS:
        out[f'pos_{y[2:-4]}'] = g[y].sum().astype(int)
        out[f'pct_{y[2:-4]}'] = (100 * g[y].mean()).round(2)
    return out
per_split = rates(land, 'split')
print(per_split.to_string())

## 2.2 Per era and per ICU day

The era table decides which anchor-year groups train the models (`TRAIN_ERAS` in stage 3); patients anchored in 2017 to 2019 are the primary test group and those anchored in 2020 to 2022 the documentation-shift test. The split is by the patient's anchor year group, so it is patient-disjoint and only approximately temporal: a patient's later admissions keep their group whatever year they fall in (`years_after_anchor` on every row says how far; 01 table 19 tabulates it), and the test groups were inspected while the pipeline was being developed. What stage 3 reports is therefore an exploratory internal validation, not a temporal one and not an untouched one; `docs/model_findings.md` says what a stronger claim would need. The ICU-day curve shows whether the risk changes with time in the unit, which is why `icu_day` is a feature.

In [ ]:
# 2.2 Per era and per ICU day
per_era = rates(land, ['anchor_year_group', 'split'])
print(per_era.to_string())
per_day = land.groupby('icu_day', observed=True).agg(rows=('stay_id', 'size'), pct_lt54=('y_lt54_24h', lambda s: round(100 * s.mean(), 2)),
                                                     pct_gt300=('y_gt300_24h', lambda s: round(100 * s.mean(), 2)))
print(per_day.head(28).to_string())
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(per_day.index, per_day['pct_lt54'], marker='o', label='glucose < 54 mg/dL in next 24 h')
ax.plot(per_day.index, per_day['pct_gt300'], marker='s', label='glucose > 300 mg/dL in next 24 h')
ax.set_xlabel('ICU day of the landmark'); ax.set_ylabel('% of at-risk landmark rows'); ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False); plt.tight_layout(); plt.show()

## 2.3 Censoring

A label window ends at the earliest of 24 h, ICU discharge and death, so a short window has had less time to show an event. Stage 3 drops rows under 6 h as a sensitivity analysis; the share here says how much that costs.

In [ ]:
# 2.3 Short windows and their event rate
w = land['window_hours']
cens = pd.DataFrame({'rows': [len(w), int((w < 24).sum()), int((w < 12).sum()), int((w < 6).sum())],
                     'pct': [100.0] + [round(100 * (w < h).mean(), 2) for h in (24, 12, 6)]},
                    index=['all', 'window < 24 h', 'window < 12 h', 'window < 6 h'])
print(cens.to_string())
print('\nevent rate by window length (rows with a short window have less time to show an event):')
bins = pd.cut(w, [0, 6, 12, 23.999, 24.001], labels=['<6 h', '6-12 h', '12-24 h', 'full 24 h'])
print(land.groupby(bins, observed=True)['y_lt54_24h'].agg(rows='size', pct_lt54=lambda s: round(100 * s.mean(), 2)).to_string())

## 2.4 Leakage assertion and key uniqueness

Every feature row carries `max_feature_time`, the latest timestamp any of its inputs came from. It must be earlier than the landmark on every row, and (stay, k) must be unique. The SQL checked this once; the notebook checks the rows it actually received.

In [ ]:
# 2.4 The leakage assertion on every row, and the key
def naive(s):
    s = pd.to_datetime(s)
    return s.dt.tz_localize(None) if getattr(s.dt, 'tz', None) is not None else s
tk, mft = naive(feat['t_k']), naive(feat['max_feature_time'])
violations = int((mft >= tk).sum())
dupes = int(feat.duplicated(['stay_id', 'k']).sum())
print(f'rows whose latest input is at or after the landmark: {violations}   duplicate (stay_id, k) keys: {dupes}')
assert violations == 0 and dupes == 0, 'the feature table is not usable as it stands'
print('closest input to a landmark (hours before t_k):', round(float(((tk - mft).dt.total_seconds() / 3600).min()), 3))

## 2.5 Missingness

Missing values are information in the ICU (no lactate drawn means nobody was worried about one), so the models keep them as missing plus an indicator rather than imputing them away. The bookkeeping columns are never model inputs; everything else is.

In [ ]:
# 2.5 Share missing per input (the roles come from src/hypo_tutorial/columns.py)
inputs = C.input_columns(feat)
print(f'{len(inputs)} model input columns, {len(C.BOOKKEEPING)} bookkeeping columns')
miss = feat[inputs].isna().mean().sort_values(ascending=False)
print('\nshare missing, the 25 most often missing inputs:')
print((100 * miss.head(25)).round(1).to_string())
fig, ax = plt.subplots(figsize=(7, 5))
top = (100 * miss.head(25))[::-1]
ax.barh(top.index, top.values); ax.set_xlabel('% of rows missing'); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 2.6 Monitoring density

`label_window_n_glucose` counts the glucose results inside the label window. It is future information and never a feature, but it says whether a negative label means "no event" or "nobody checked", which is the first question a clinician asks about this outcome.

In [ ]:
# 2.6 Glucose results in the label window, by drip status
n = feat['label_window_n_glucose']
dens = feat.assign(bucket=pd.cut(n, [-1, 0, 1, 3, 1e9], labels=['0 results', '1', '2-3', '4+'])) \
           .groupby(['drip_running_at_tk', 'bucket'], observed=True).size().unstack(fill_value=0)
dens['rows'] = dens.sum(axis=1)
print((dens.div(dens['rows'], axis=0).drop(columns='rows') * 100).round(1).assign(rows=dens['rows']).to_string())

## 2.7 Save the cohort tables

The tables of this stage, as CSV next to the check tables, for the deck.

In [ ]:
# 2.7 Cohort tables to results/
for name, df in [('eda_per_split', per_split), ('eda_per_era', per_era), ('eda_per_icu_day', per_day), ('eda_censoring', cens),
                 ('eda_missingness', (100 * miss).round(2).rename('pct_missing').to_frame()), ('eda_monitoring_density', dens)]:
    df.to_csv(RESULTS / f'{name}.csv')
print('written to', RESULTS)

## 3. The models: two tabular arms, two outcomes, two test eras

For each outcome, a logistic regression (arm A) and a gradient-boosted tree model (arm B) trained on the landmark rows of patients anchored in 2008 to 2016 and scored on those anchored in 2017 to 2019 (the primary test group) and in 2020 to 2022 (the documentation-shift test, kept apart because three fifths of its ICU stays have no flowsheet medication or procedure data; `docs/cohort_findings.md`, tables 14, 16 and 17). The code is `src/hypo_tutorial/` (column roles, feature preparation, the two arms, the evaluation); the notebook calls it and prints what it did. Outputs: aggregate tables and figures under `results/`, row-level predictions under `predictions/` next to the parquet, which hold MIMIC-IV identifiers and stay on your own Drive.

## 3.1 The rows, their roles, and the split

`split` is assigned by two DECLAREs in `01_cohort_labels.sql`: `test` is 2017 to 2019 and `test_shifted` is 2020 to 2022; the rest is `train`. The training eras can be narrowed here (`TRAIN_ERAS`); the sensitivity section drops 2008 to 2010, the one era whose low-glucose rate stands apart (1.64% against 1.0 to 1.2%).

The column roles are written out in `src/hypo_tutorial/columns.py` and checked here: a column the SQL adds later is refused until someone gives it a role.

In [ ]:
# 3.1 Load, check the roles, choose the training eras
feat = pd.read_parquet(PARQUET / 'features.parquet')
# a fixed row order: the subject-grouped folds follow the order of the rows, and a file pulled twice from BigQuery came
# back with its rows shuffled, which moved the trees and the third decimal of the AUROC between two runs on the same
# data (releases .12 and .14). Sorted on the key, the folds, the models and every number below depend on the data alone.
feat = feat.sort_values(['subject_id', 'stay_id', 'k'], kind='mergesort').reset_index(drop=True)
C.check_roles(feat)
LABELS, INPUTS = C.LABELS, C.input_columns(feat)
TRAIN_ERAS = ['2008 - 2010', '2011 - 2013', '2014 - 2016']           # the notebook setting the README names
assert set(feat.loc[feat['split'] == 'train', 'anchor_year_group']) <= set(TRAIN_ERAS) | set(), 'TRAIN_ERAS must cover the train split or narrow it on purpose'

SMOKE = len(feat) < 20_000            # a development sample under 20,000 rows: 100 bootstrap draws instead of 1,000, same code path
N_BOOT = 100 if SMOKE else 1000
print(f'{len(feat):,} rows, {feat.subject_id.nunique():,} subjects, {len(INPUTS)} input columns, {len(C.BOOKKEEPING)} bookkeeping columns' + (' | SMOKE run' if SMOKE else ''))

splits = {'train': feat[(feat.split == 'train') & feat.anchor_year_group.isin(TRAIN_ERAS)],
          'test': feat[feat.split == 'test'], 'test_shifted': feat[feat.split == 'test_shifted']}
summary = pd.DataFrame({name: {'rows': len(d), 'subjects': d.subject_id.nunique(), 'eras': ', '.join(sorted(d.anchor_year_group.unique())),
                               **{f'events {o}': int(d[y].sum()) for o, y in C.OUTCOMES.items()},
                               **{f'rate {o} %': round(100 * d[y].mean(), 2) for o, y in C.OUTCOMES.items()}}
                        for name, d in splits.items()}).T
print(summary.to_string())
summary.to_csv(RESULTS / 'model_splits.csv')

## 3.2 Feature preparation, fitted on the training rows

Winsorising at the training 0.1 and 99.9 percentiles, missing indicators, medians for the logistic arm, one-hot categories with rare levels pooled, standardisation for the logistic arm. The tree arm receives the winsorised values with their missing values intact, because LightGBM routes a missing value down its own branch and the indicator would be redundant. The printed table is the audit trail: every cut-point a reader could check against the training rows.

In [ ]:
# 3.2 Fit the preparation on the training rows only, then transform every split with it (inside the cross-validation
#     the preparation is refitted per fold, models.py; this one serves the final models and the test eras)
prep = F.FeaturePrep().fit(splits['train'])
X = {}
for name, d in splits.items():
    X[name] = {'lr': prep.transform(d, for_tree=False), 'tree': prep.transform(d, for_tree=True)}
print(f'{len(prep.columns_)} matrix columns: {len(prep.numeric_)} numeric, {len(prep.binary_)} binary, '
      f'{len(prep.indicator_cols_)} missing indicators, {sum(len(v) + 1 for v in prep.categories_.values())} one-hot levels')
prep_summary = prep.summary()
prep_summary.to_csv(RESULTS / 'feature_prep_cutpoints.csv', index=False)
print(prep_summary[prep_summary.column.isin(['glucose_last', 'lactate_last', 'insulin_bolus_units_24h', 'hr_max_24h', 'creatinine_last'])].to_string(index=False))

## 3.3 The two arms

**Arm A, logistic regression.** One knob, the inverse L2 penalty `C`, chosen on grouped cross-validated log loss. Coefficients are on standardised inputs, so their size is comparable.

**Arm B, LightGBM.** Fixed conservative settings (learning rate 0.03, 31 leaves, at least 100 rows per leaf, 80% feature and row subsampling, L2 of 1); the number of trees comes from early stopping on each fold's validation log loss, and the final model is refit on all training rows with the mean best iteration. Nothing is tuned on a test era.

Both arms produce out-of-fold predictions for every training row. Those feed the Platt recalibration (a logistic regression of the outcome on the model's logit) that is applied to the test eras alongside the raw model; a slope near 1 and an intercept near 0 say the raw model was already calibrated. The raw models are the primary result; the recalibrated rows are exploratory (the out-of-fold predictions are shaped by the same folds that chose `C` and the stopping round, `src/hypo_tutorial/models.py`), and a nested scheme would be needed before a calibrated prediction could be the main result.

In [ ]:
# 3.3 Fit both arms for one outcome: returns the models, the out-of-fold predictions and the tuning tables
def fit_arms(outcome):
    y_col = C.OUTCOMES[outcome]
    tr = splits['train']
    y, groups = tr[y_col].to_numpy().astype(int), tr['subject_id'].to_numpy()
    # the preparation is refitted inside every fold (models.py); the full-train matrices feed the final models only
    lr, oof_lr, path_lr = M.fit_logistic(tr, y, groups, F.FeaturePrep, X['train']['lr'])
    gbm, oof_gbm, folds_gbm = M.fit_lightgbm(tr, y, groups, F.FeaturePrep, X['train']['tree'])
    platt = {'A': M.Platt().fit(oof_lr, y), 'B': M.Platt().fit(oof_gbm, y)}
    print(f'[{outcome}] arm A: C = {lr.C} (CV log loss path below); arm B: {int(folds_gbm.final_rounds.iloc[0])} trees '
          f'(per-fold best {folds_gbm.best_iteration.tolist()}); Platt on OOF: A slope {platt["A"].slope_:.2f} intercept {platt["A"].intercept_:.2f}, '
          f'B slope {platt["B"].slope_:.2f} intercept {platt["B"].intercept_:.2f}')
    print(path_lr.to_string(index=False))
    path_lr.to_csv(RESULTS / f'cv_path_armA_{outcome}.csv', index=False)
    folds_gbm.to_csv(RESULTS / f'cv_folds_armB_{outcome}.csv', index=False)
    # how the out-of-fold predictions themselves were calibrated, and what Platt learned from them: the first real
    # run showed B+Platt over-predicting the primary test era by 16% while raw B was calibrated, so this table is
    # the evidence for whether recalibration from out-of-fold predictions should be trusted at all
    oof_rows = []
    for arm, oof in [('A', oof_lr), ('B', oof_gbm)]:
        s, a = E.calibration_slope_intercept(y, oof)
        oof_rows.append({'outcome': outcome, 'arm': arm, 'oof_auroc': E.auroc(y, oof), 'oof_oe_ratio': E.oe_ratio(y, oof),
                         'oof_cal_slope': s, 'oof_cal_intercept': a, 'platt_slope': platt[arm].slope_, 'platt_intercept': platt[arm].intercept_,
                         'mean_oof_prediction': float(np.mean(oof)), 'train_event_rate': float(y.mean())})
    pd.DataFrame(oof_rows).to_csv(RESULTS / f'oof_calibration_{outcome}.csv', index=False)
    return {'A': lr, 'B': gbm}, {'A': oof_lr, 'B': oof_gbm}, platt

fits = {o: fit_arms(o) for o in C.OUTCOMES}

## 3.4 Evaluation on the two test eras

For each arm, outcome and era: AUROC, AUPRC, Brier score, O/E ratio, calibration slope and intercept, each with a subject-level bootstrap interval; the reliability table behind the calibration plot; the decision curve over thresholds 0.5 to 10%; and the operating points when the top 5, 10 and 20% of rows by risk are flagged. The recalibrated predictions are reported in the same tables under `arm A+Platt` and `arm B+Platt`.

The 16 bootstraps below (2 outcomes x 2 arms x 2 eras x raw / recalibrated) each draw subjects with replacement 1,000 times. A draw is computed as integer row weights over the rows sorted once by predicted risk, which is the same arithmetic as stacking the drawn rows but about twenty times faster (`src/hypo_tutorial/evaluate.py`, `tests/test_evaluate.py` show the two agree to floating point). The point estimates are scikit-learn's own functions. Everything in this notebook runs on the CPU; a GPU runtime would not change the time.

In [ ]:
# 3.4 Score every split, evaluate, save. Row-level predictions go to PREDICTIONS (private); everything in RESULTS is aggregate.
PALETTE = {'A': '#2a78d6', 'B': '#eb6834', 'ref': '#8a8984'}      # fixed hue per arm, gray for references
# decision-curve thresholds follow the outcome's base rate: an alert for a 1% event is useful at 0.5 to 10%,
# one for a 10% event at 2 to 40%; a curve drawn below the prevalence only shows that acting on everyone wins
THRESH = {'lt54': E.THRESHOLDS, 'gt300': np.round(np.arange(0.02, 0.401, 0.02), 3)}
rows, reliability, curves, operating, subgroups = [], [], [], [], []
preds = []
for outcome, (models, oof, platt) in fits.items():
    y_col = C.OUTCOMES[outcome]
    for arm, model in models.items():
        for era in ['test', 'test_shifted']:
            d = splits[era]
            p_raw = M.predict(model, X[era]['lr' if arm == 'A' else 'tree'])
            for variant, p in [(f'{arm}', p_raw), (f'{arm}+Platt', platt[arm].transform(p_raw))]:
                y, g = d[y_col].to_numpy().astype(int), d['subject_id'].to_numpy()
                boot = E.bootstrap(y, p, g, n_boot=N_BOOT)
                for _, r in boot.iterrows():
                    rows.append({'outcome': outcome, 'arm': variant, 'era': era, **r.to_dict(), 'rows': len(y), 'events': int(y.sum())})
                rows.append({'outcome': outcome, 'arm': variant, 'era': era, 'metric': 'ece', 'estimate': E.ece(y, p), 'ci_low': np.nan, 'ci_high': np.nan, 'rows': len(y), 'events': int(y.sum())})
                reliability.append(E.reliability_table(y, p).assign(outcome=outcome, arm=variant, era=era))
                curves.append(E.net_benefit(y, p, THRESH[outcome]).assign(outcome=outcome, arm=variant, era=era))
                operating.append(E.operating_points(y, p).assign(outcome=outcome, arm=variant, era=era))
            # subgroups on the raw predictions of each arm
            # subgroups on the raw predictions of each arm. Race is collapsed to MIMIC's coarse groups (E.race_group), units and
            # other levels under 100 rows are pooled, and cells under 10 events carry no count: the table is written to be shared
            sg = d[['subject_id', 'stay_id', 'k', y_col, 'drip_running_at_tk', 'anchor_year_group', 'first_careunit', 'gender', 'race', 'drip_emar_24h', 'drip_inputevents_24h']].copy()
            sg['p'] = p_raw
            sg['icu_day_bucket'] = E.icu_day_bucket(sg['k']).to_numpy()
            sg['drip_record'] = np.select([sg.drip_inputevents_24h == 1, sg.drip_emar_24h == 1], ['flowsheet', 'eMAR only'], 'no drip')
            sg['race_group'] = E.race_group(sg['race']).to_numpy()
            for by in ['drip_running_at_tk', 'drip_record', 'anchor_year_group', 'first_careunit', 'gender', 'race_group', 'icu_day_bucket']:
                subgroups.append(E.subgroup_table(sg, y_col, 'p', by).rename(columns={by: 'level'}).assign(subgroup=by, outcome=outcome, arm=arm, era=era))
            preds.append(d[['subject_id', 'hadm_id', 'stay_id', 'k', 'split', 'anchor_year_group', y_col]].assign(outcome=outcome, arm=arm, p_raw=p_raw, p_platt=platt[arm].transform(p_raw)))

metrics = pd.DataFrame(rows)
metrics.to_csv(RESULTS / 'metrics_by_arm_outcome_era.csv', index=False)
pd.concat(reliability).to_csv(RESULTS / 'reliability_tables.csv', index=False)
pd.concat(curves).to_csv(RESULTS / 'decision_curves.csv', index=False)
pd.concat(operating).to_csv(RESULTS / 'operating_points.csv', index=False)
pd.concat(subgroups).to_csv(RESULTS / 'subgroups.csv', index=False)
pd.concat(preds).to_parquet(PREDICTIONS / 'predictions_tabular_arms.parquet', index=False)   # row-level: stays private

# the headline table: one line per arm, outcome and era
def fmt(r):
    return f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]" if np.isfinite(r.ci_low) else f"{r.estimate:.3f}"
head = metrics[metrics.metric.isin(['auroc', 'auprc', 'brier', 'oe_ratio', 'cal_slope', 'cal_intercept', 'ece'])].copy()
head['value'] = head.apply(fmt, axis=1)
headline = head.pivot_table(index=['outcome', 'era', 'arm'], columns='metric', values='value', aggfunc='first')[['auroc', 'auprc', 'brier', 'oe_ratio', 'cal_slope', 'cal_intercept', 'ece']]
headline.to_csv(RESULTS / 'headline_metrics.csv')
with pd.option_context('display.width', 250, 'display.max_colwidth', 40):
    print(headline.to_string())

## 3.5 Figures

Reliability plots (observed against predicted risk per decile) and decision curves for the primary outcome on both eras, ROC and precision-recall curves, and the feature importance of each arm: standardised coefficients for arm A, mean absolute SHAP values for arm B. One hue per arm throughout (blue A, orange B); references in gray.

In [ ]:
# 3.5 Figures: calibration, decision curves, ROC / PR, importance
from sklearn.metrics import roc_curve, precision_recall_curve
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': '#e5e4e0', 'grid.linewidth': 0.6, 'font.size': 10})
rel_all, curves_all = pd.concat(reliability), pd.concat(curves)

def calibration_and_decision(outcome):
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    for j, era in enumerate(['test', 'test_shifted']):
        ax = axes[0, j]
        for arm in ['A', 'B']:
            t = rel_all[(rel_all.outcome == outcome) & (rel_all.arm == arm) & (rel_all.era == era)]
            ax.plot(t.mean_predicted, t.observed, marker='o', ms=4, lw=1.5, color=PALETTE[arm], label=f'arm {arm}')
        lim = max(rel_all[(rel_all.outcome == outcome)][['mean_predicted', 'observed']].max().max() * 1.05, 0.01)
        ax.plot([0, lim], [0, lim], color=PALETTE['ref'], lw=1, ls='--', label='perfect calibration')
        ax.set_xlim(0, lim); ax.set_ylim(0, lim)
        ax.set_xlabel('mean predicted risk (decile)'); ax.set_ylabel('observed event rate')
        ax.set_title(f'{outcome}: calibration, {era}'); ax.legend(frameon=False)
        ax = axes[1, j]
        for arm in ['A', 'B']:
            t = curves_all[(curves_all.outcome == outcome) & (curves_all.arm == arm) & (curves_all.era == era)]
            ax.plot(100 * t.threshold, t.net_benefit_model, lw=1.5, color=PALETTE[arm], label=f'arm {arm}')
        t = curves_all[(curves_all.outcome == outcome) & (curves_all.arm == 'A') & (curves_all.era == era)]
        ax.plot(100 * t.threshold, t.net_benefit_all, color=PALETTE['ref'], lw=1, ls='--', label='act on every row')
        ax.axhline(0, color=PALETTE['ref'], lw=1, ls=':', label='act on no row')
        top = max(t.net_benefit_all.max(), curves_all[(curves_all.outcome == outcome)].net_benefit_model.max())
        ax.set_ylim(-0.25 * top, 1.1 * top)        # the treat-all line runs far negative; the model curves are the point
        ax.set_xlabel('threshold probability (%)'); ax.set_ylabel('net benefit')
        ax.set_title(f'{outcome}: decision curve, {era}'); ax.legend(frameon=False)
    plt.tight_layout(); fig.savefig(RESULTS / 'figures' / f'calibration_decision_{outcome}.png', bbox_inches='tight'); plt.show()

def roc_pr(outcome):
    y_col = C.OUTCOMES[outcome]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    pr_curves = []
    for arm in ['A', 'B']:
        for era, ls in [('test', '-'), ('test_shifted', ':')]:
            d = splits[era]; y = d[y_col].to_numpy().astype(int)
            p = M.predict(fits[outcome][0][arm], X[era]['lr' if arm == 'A' else 'tree'])
            if 0 < y.sum() < len(y):
                fpr, tpr, _ = roc_curve(y, p); axes[0].plot(fpr, tpr, ls=ls, lw=1.5, color=PALETTE[arm], label=f'arm {arm}, {era}')
                prec, rec, _ = precision_recall_curve(y, p); axes[1].plot(rec, prec, ls=ls, lw=1.5, color=PALETTE[arm], label=f'arm {arm}, {era}'); pr_curves.append((prec, rec))
    axes[0].plot([0, 1], [0, 1], color=PALETTE['ref'], lw=1, ls='--'); axes[0].set_xlabel('false positive rate'); axes[0].set_ylabel('sensitivity'); axes[0].set_title(f'{outcome}: ROC')
    # the first few rows of a precision-recall curve are a noisy spike at PPV 1.0 that hides the curve: scale the axis to the
    # curve past 2% recall, and draw the prevalence, which is the PPV of acting on everyone
    top = max(prec_[rec_ > 0.02].max() for prec_, rec_ in pr_curves) if pr_curves else 1.0
    axes[1].set_ylim(0, min(1.0, 1.15 * top))
    for era, ls in [('test', '-'), ('test_shifted', ':')]:
        axes[1].axhline(splits[era][y_col].mean(), color=PALETTE['ref'], lw=1, ls=ls, label=f'prevalence, {era}')
    axes[1].set_xlabel('sensitivity (recall)'); axes[1].set_ylabel('positive predictive value'); axes[1].set_title(f'{outcome}: precision-recall')
    for ax in axes: ax.legend(frameon=False, fontsize=8)
    plt.tight_layout(); fig.savefig(RESULTS / 'figures' / f'roc_pr_{outcome}.png', bbox_inches='tight'); plt.show()

for outcome in C.OUTCOMES:
    calibration_and_decision(outcome); roc_pr(outcome)

In [ ]:
# 3.6 What the models use: standardised coefficients (arm A) and mean |SHAP| (arm B), primary outcome first
importance = {}
for outcome, (models, _, _) in fits.items():
    coef = M.logistic_coefficients(models['A'], prep.columns_, top=20)
    shap_summary, shap_values, Xs = M.lightgbm_shap(models['B'], X['test']['tree'], top=20, sample=2000 if SMOKE else 5000)
    coef.to_csv(RESULTS / f'importance_armA_{outcome}.csv', index=False); shap_summary.to_csv(RESULTS / f'importance_armB_{outcome}.csv', index=False)
    importance[outcome] = (coef, shap_summary)
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    c = coef.iloc[::-1]; axes[0].barh(c.feature, c.standardised_coefficient, color=PALETTE['A']); axes[0].axvline(0, color=PALETTE['ref'], lw=1)
    axes[0].set_title(f'{outcome}: arm A, standardised coefficients (top 20)'); axes[0].grid(axis='y', visible=False)
    s = shap_summary.iloc[::-1]; axes[1].barh(s.feature, s.mean_abs_shap, color=PALETTE['B'])
    axes[1].set_title(f'{outcome}: arm B, mean |SHAP| on the test era (top 20)'); axes[1].grid(axis='y', visible=False)
    plt.tight_layout(); fig.savefig(RESULTS / 'figures' / f'importance_{outcome}.png', bbox_inches='tight'); plt.show()

## 3.6 Subgroups and operating points

Discrimination by drip status, by the record that saw the drip (flowsheet, eMAR only, none), by era, care unit, sex, race and ICU day, for each arm and era. A cell with fewer than 10 events shows its counts and no metric. The operating-point table says what a unit would see: flag the top 5% of rows each day and you catch this share of events at this positive predictive value.

In [ ]:
# 3.7 Subgroup and operating-point tables for the primary outcome, arm B, both eras (the CSVs hold every combination)
sub_all, op_all = pd.concat(subgroups), pd.concat(operating)
with pd.option_context('display.width', 250, 'display.max_rows', 200):
    show = sub_all[(sub_all.outcome == 'lt54') & (sub_all.arm == 'B') & (sub_all.subgroup.isin(['drip_running_at_tk', 'drip_record', 'icu_day_bucket', 'gender']))]
    print(show[['era', 'subgroup', 'level', 'rows', 'events', 'event_rate', 'auroc', 'auprc', 'oe_ratio']].round(3).to_string(index=False))
    print()
    print(op_all[(op_all.outcome == 'lt54') & (op_all.arm.isin(['A', 'B']))][['era', 'arm', 'alert_rate', 'alerts_per_100_rows', 'sensitivity', 'ppv', 'events_caught', 'events_total']].round(3).to_string(index=False))

## 3.7 Sensitivities

Five questions a reviewer would ask, each answered by refitting or re-scoring arm B:

1. **Training eras.** Drop 2008 to 2010, whose low-glucose rate (1.64%) stands apart from the later eras (1.0 to 1.2%). Does the test-era AUROC move?
2. **Measured rows only.** Score only the test rows with at least one glucose result in the label window (`label_window_n_glucose > 0`): a row with no result cannot be a positive, and the 5 to 6% of such rows dilute the event rate.
3. **Windows of 6 h or more.** Score only the rows whose label window is at least 6 h; a row censored by discharge or death after two hours had little chance to show an event.
4. **Anchor-year admissions only.** The split is by the patient's anchor year group, and a patient's later admissions keep the group whatever calendar year they fall in. Score only the test rows whose admission is in the anchor year itself (`years_after_anchor = 0`), the rows whose calendar period is the group's own three years. This is a check on the test rows alone: the training rows keep their later admissions, so it does not turn the split into a temporal one.
5. **The repeat-excluded endpoint.** Refit arm B on `y_lt54_excl_24h`, the < 54 outcome after the CMS816 repeat-result exclusion (the definition releases .20 and .21 used as primary), and score it against that endpoint on both test groups. The primary endpoint keeps every documented low; this row says what the exclusion changes.

In [ ]:
# 3.8 Sensitivities on arm B, primary outcome
y_col = C.OUTCOMES['lt54']
sens = []
def auroc_ci(y, p, g):
    b = E.bootstrap(y, p, g, n_boot=N_BOOT, metrics={'auroc': E.auroc, 'auprc': E.auprc})
    b = b[b.metric.isin(['auroc', 'auprc'])]
    return {m: (r.estimate, r.ci_low, r.ci_high) for m, r in zip(b.metric, b.itertuples())}

# (1) without the first era: refit the preparation and arm B on 2011-2016
tr2 = splits['train'][splits['train'].anchor_year_group != '2008 - 2010']
if len(tr2) > 100 and tr2[y_col].sum() >= 5:
    prep2 = F.FeaturePrep().fit(tr2)
    gbm2, _, _ = M.fit_lightgbm(tr2, tr2[y_col].to_numpy().astype(int), tr2.subject_id.to_numpy(), F.FeaturePrep, prep2.transform(tr2, for_tree=True))
    for era in ['test', 'test_shifted']:
        d = splits[era]; r = auroc_ci(d[y_col].to_numpy().astype(int), M.predict(gbm2, prep2.transform(d, for_tree=True)), d.subject_id.to_numpy())
        sens.append({'sensitivity': 'train on 2011-2016 only', 'era': era, 'rows': len(d), 'events': int(d[y_col].sum()), **{f'{m}': v[0] for m, v in r.items()}, **{f'{m}_ci': f'[{v[1]:.3f}, {v[2]:.3f}]' for m, v in r.items()}})
# (2), (3) and (4): re-score the fitted arm B on subsets of the test rows
gbm = fits['lt54'][0]['B']
for label, mask_fn in [('measured rows only', lambda d: d.label_window_n_glucose > 0), ('windows of 6 h or more', lambda d: d.window_hours >= 6),
                       ('anchor-year admissions only', lambda d: d.years_after_anchor == 0)]:
    for era in ['test', 'test_shifted']:
        d = splits[era]; keep = mask_fn(d).to_numpy()
        p = M.predict(gbm, X[era]['tree'])[keep]; y = d[y_col].to_numpy().astype(int)[keep]
        r = auroc_ci(y, p, d.subject_id.to_numpy()[keep])
        sens.append({'sensitivity': label, 'era': era, 'rows': int(keep.sum()), 'events': int(y.sum()), **{f'{m}': v[0] for m, v in r.items()}, **{f'{m}_ci': f'[{v[1]:.3f}, {v[2]:.3f}]' for m, v in r.items()}})
# (5) the repeat-excluded endpoint: refit arm B on y_lt54_excl_24h (same training rows, same preparation) and score it against
#     that endpoint; the row is read against the main analysis, whose endpoint keeps every documented low
y_alt = 'y_lt54_excl_24h'
tr = splits['train']
if tr[y_alt].sum() >= 5:
    gbm_alt, _, _ = M.fit_lightgbm(tr, tr[y_alt].to_numpy().astype(int), tr.subject_id.to_numpy(), F.FeaturePrep, X['train']['tree'])
    for era in ['test', 'test_shifted']:
        d = splits[era]; y = d[y_alt].to_numpy().astype(int)
        r = auroc_ci(y, M.predict(gbm_alt, X[era]['tree']), d.subject_id.to_numpy())
        sens.append({'sensitivity': 'repeat-excluded < 54 endpoint (refit)', 'era': era, 'rows': len(d), 'events': int(y.sum()), **{f'{m}': v[0] for m, v in r.items()}, **{f'{m}_ci': f'[{v[1]:.3f}, {v[2]:.3f}]' for m, v in r.items()}})
base = metrics[(metrics.outcome == 'lt54') & (metrics.arm == 'B') & (metrics.metric.isin(['auroc', 'auprc']))]
for era in ['test', 'test_shifted']:
    b = base[base.era == era].set_index('metric')
    sens.insert(0 if era == 'test' else 1, {'sensitivity': 'main analysis', 'era': era, 'rows': int(b.loc['auroc', 'rows']), 'events': int(b.loc['auroc', 'events']),
                                             'auroc': b.loc['auroc', 'estimate'], 'auprc': b.loc['auprc', 'estimate'],
                                             'auroc_ci': f"[{b.loc['auroc', 'ci_low']:.3f}, {b.loc['auroc', 'ci_high']:.3f}]", 'auprc_ci': f"[{b.loc['auprc', 'ci_low']:.3f}, {b.loc['auprc', 'ci_high']:.3f}]"})
sens = pd.DataFrame(sens)
sens.to_csv(RESULTS / 'sensitivities_armB_lt54.csv', index=False)
print(sens.round(3).to_string(index=False))

## 3.8 What to report

Everything under `results/` is an aggregate and can be shared as it is: `headline_metrics.csv` (the table above), `metrics_by_arm_outcome_era.csv` (the same with every bootstrap interval), `subgroups.csv`, `operating_points.csv`, `decision_curves.csv`, `reliability_tables.csv`, `sensitivities_armB_lt54.csv`, `oof_calibration_*.csv` (how the out-of-fold predictions were calibrated and what Platt learned from them), the tuning tables (`cv_path_armA_*.csv`, `cv_folds_armB_*.csv`), `feature_prep_cutpoints.csv`, `model_splits.csv`, `run_settings.json` and the figures under `figures/`. Send the whole folder. `predictions/predictions_tabular_arms.parquet` holds one row per landmark with identifiers and stays on your Drive.

Three readings to carry into the deck, from the release .22 run (12 October 2026; `docs/model_findings.md` has every run's tables, and `docs/model_results/` the files the deck is generated from). First, the 2020 to 2022 column is a test of the model under a documentation shift, not of the patients: discrimination held (arm B AUROC 0.852 against 0.863 on 2017 to 2019) while both arms under-predicted the group by 12% (O/E 1.12, an interval that includes 1), most of it on the rows with an infusion running (O/E 1.51), whose flowsheet features are missing or truncated there. Second, the raw LightGBM was already calibrated on the primary test group (O/E 0.95, slope 1.07), and the Platt recalibration fitted on the out-of-fold predictions made it worse (O/E 0.85): a recalibration learned on one set of models and applied to another needs its own check, which `oof_calibration_*.csv` provides (the fold models under-predict the rows they never saw by 12%). Third, the global AUROC hides the subgroup that matters most: among rows with an insulin infusion running, arm B's AUROC was 0.73 on the primary test group (37 events), so the model ranks risk less well exactly where iatrogenic hypoglycemia is most likely. The deck is built from these files by a script, so a number on a slide can always be traced to a file in this folder.

In [ ]:
# 3.9 Versions, for the README's reproducibility line
import sklearn, lightgbm, shap
versions = {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__, 'scikit-learn': sklearn.__version__,
            'lightgbm': lightgbm.__version__, 'shap': shap.__version__, 'matplotlib': matplotlib.__version__}
print(versions)
json.dump({'release': RELEASE, 'rows': int(len(feat)), 'train_eras': TRAIN_ERAS, 'n_boot': N_BOOT, 'smoke': bool(SMOKE), 'versions': versions},
          open(RESULTS / 'run_settings.json', 'w'), indent=1)

## 4. Pack the results for sharing

One zip of the `results/` folder (tables and figures, all aggregates), written next to it on Drive and offered for download here. Downloading the folder from the Drive website before Drive has finished syncing gives an empty `figures/` folder, which is what happened on the first real run; this cell reads the files from the runtime, so it cannot. The zip holds the SQL check tables of the data stage, the cohort tables of stage 2 and the model results of stage 3 together. The `parquet/` and `predictions/` folders are never packed.

In [ ]:
# 4. Zip results/ (aggregates only) next to it and hand the zip over; parquet/ and predictions/ stay where they are
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
local = pathlib.Path('/content' if IN_COLAB else str(RESULTS.parent)) / f'results_{RELEASE}_{stamp}'
archive = shutil.make_archive(str(local), 'zip', root_dir=RESULTS.parent, base_dir=RESULTS.name)   # built on the runtime's disk
n_files = sum(1 for _ in pathlib.Path(RESULTS).rglob('*') if _.is_file())
print(f'{archive}: {n_files} files, {os.path.getsize(archive) / 1e6:.1f} MB')
if IN_COLAB:
    from google.colab import files, drive
    shutil.copy(archive, RESULTS.parent / os.path.basename(archive))   # a copy on Drive next to results/
    drive.flush_and_unmount()                 # the tables, figures and the zip are on Drive before anything else reads them
    files.download(archive)                   # and the local zip comes to your computer now, without waiting for Drive to sync